In [1]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score # t9sim data
from sklearn.datasets import make_regression # dyal dataset
from sklearn.metrics import *
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
import joblib
import sklearn.linear_model as model
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

In [2]:
# # L PIPELINE
# cleaned_dataset = pd.read_csv('./cleaned_dataset.csv')
# train_test_id = np.int16(cleaned_dataset.shape[0] * 0.8)
# cleaned_data = cleaned_dataset.to_numpy()
# x_train, y_train = cleaned_data[:train_test_id, :-1], cleaned_data[:train_test_id, -1:]
# x_test, y_test = cleaned_data[train_test_id:, :-1], cleaned_data[train_test_id:, -1:]

In [3]:
# train_df = pd.read_csv('./train_dataset.csv')
# test_df = pd.read_csv('./test_dataset.csv')
full_df = pd.read_csv('./transformed_df.csv')
full = full_df.to_numpy()
full_x = full[:, :-1]
full_y = full[:, -1:]
train_inputs, test_inputs, train_outputs, test_outputs = train_test_split(full_x, full_y, test_size=0.2, random_state=1)

In [4]:
linear_regression = model.LinearRegression(fit_intercept=False)
linear_regression.fit(train_inputs, train_outputs)
y_pred = linear_regression.predict(test_inputs)

In [5]:
def mse_rmse(y_true:np.ndarray, y_pred:np.ndarray):
  mae = np.abs(y_pred - y_true).mean()
  mse = (np.subtract(y_pred, y_true) ** 2).mean()
  rmse = np.sqrt(mse)
  r_square = r2_score(y_true, y_pred)
  return mae, rmse, r_square

In [6]:
def write_line(line):
  path = './benchmark.txt'
  if not os.path.exists(path):
    os.mknod(path)
  with open(path, 'a', encoding='utf-8') as f:
    f.write(line + '\n')

In [7]:
mae, rmse, r_square = mse_rmse(test_outputs, y_pred)
print(f'[+] MAE: {mae} | RMSE: {rmse} | R² => {r_square}')
write_line(f'[+] Linear Regression: MAE => {mae} | RMSE => {rmse} | R² => {r_square}')
joblib.dump(linear_regression, 'linear_regression.pkl')

[+] MAE: 0.016657428485497703 | RMSE: 0.04587251701438012 | R² => 0.708116794703661


['linear_regression.pkl']

In [8]:
rfr = RandomForestRegressor(random_state=1)
rfr.fit(train_inputs, train_outputs)

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestRegressor(random_state=1)

In [9]:
mae, rmse, r_square = mse_rmse(test_outputs, y_pred)
print(f'[+] MAE: {mae} | RMSE: {rmse} | R² => {r_square}')
write_line(f'[+] Random Forest: MAE => {mae} | RMSE => {rmse} | R² => {r_square}')
joblib.dump(linear_regression, 'random_forest.pkl')

[+] MAE: 0.016657428485497703 | RMSE: 0.04587251701438012 | R² => 0.708116794703661


['random_forest.pkl']

In [10]:
linear_regression2 = model.LinearRegression()
kfold = KFold(10)
linReg_grid_search = GridSearchCV(linear_regression2, {
    'fit_intercept': [False, True]
}, cv=kfold)

In [11]:
linReg_grid_search.fit(train_inputs, train_outputs)
linReg_grid_search.score(test_inputs, test_outputs)

0.7368244804929566

In [12]:
linReg_grid_search.best_params_

{'fit_intercept': True}

In [13]:
rfr2 = RandomForestRegressor()
rfr2_grid_search = GridSearchCV(rfr2, {
    'n_estimators': [100, 200],
    'criterion': ['squared_error', 'absolute_error', 'friedman_mse'],
    'min_samples_split': np.arange(2, 11, 1),
    'max_features': np.arange(1, 11, 1),
    'max_leaf_nodes': np.arange(1, 11, 1),
}, cv=kfold)

In [ ]:
rfr2_grid_search.fit(train_inputs, train_outputs)
rfr2_grid_search.score(test_inputs, test_outputs)

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please cha